# Letter-to-Word Generation: LSTM Model Training

This notebook trains an LSTM neural network to predict target words from sequences of input letters. The pipeline includes data loading, vocabulary setup, data encoding, model training, and evaluation.

In [1]:
# Load the processed dataset
import pandas as pd
df = pd.read_csv("processed_dataset.csv")

## 1. Data Loading and Preparation

Load the processed dataset and split it into training, validation, and test sets.

In [2]:
# Split dataset: 70% train, 15% validation, 15% test
from sklearn.model_selection import train_test_split

train, temp = train_test_split(df, test_size=0.3, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

# Save splits to CSV files
train.to_csv("train.csv", index=False)
val.to_csv("val.csv", index=False)
test.to_csv("test.csv", index=False)

In [3]:
# Display sample data
print(train.head())
print(train.iloc[0]["input"], "→", train.iloc[0]["target"])

                                       input_letters      input    target  \
38279                                ['A', 'R', 'Y']        ARY      MARY   
78121            ['C', 'L', 'A', 'N', 'I', 'N', 'G']    CLANING  CLEANING   
66971            ['R', 'E', 'L', 'T', 'I', 'V', 'E']    RELTIVE  RELATIVE   
18357  ['I', 'N', 'V', 'V', 'O', 'L', 'V', 'E', 'D']  INVVOLVED  INVOLVED   
79486       ['C', 'H', 'A', 'N', 'N', 'E', 'L', 'I']   CHANNELI  CHANNELS   

                            output_real_letters  
38279                      ['M', 'A', 'R', 'Y']  
78121  ['C', 'L', 'E', 'A', 'N', 'I', 'N', 'G']  
66971  ['R', 'E', 'L', 'A', 'T', 'I', 'V', 'E']  
18357  ['I', 'N', 'V', 'O', 'L', 'V', 'E', 'D']  
79486  ['C', 'H', 'A', 'N', 'N', 'E', 'L', 'S']  
ARY → MARY


## 2. Vocabulary Setup

Create letter and word to index mappings for embedding lookup during model training.

In [4]:
# Define letter vocabulary and padding token
LETTERS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
PAD_TOKEN = "<PAD>"

In [5]:
# Create letter-to-index mapping (PAD=0, A=1, B=2, ..., Z=26)
letter_to_idx = {PAD_TOKEN: 0}
for i, l in enumerate(LETTERS, start=1):
    letter_to_idx[l] = i

In [6]:
# Function to encode letter sequences to fixed-length integer sequences
def encode_letters(letters, max_len=8):
    # Convert letters to their indices
    encoded = [letter_to_idx[l] for l in letters]
    # Pad with zeros if sequence is shorter than max_len
    if len(encoded) < max_len:
        encoded += [0] * (max_len - len(encoded))
    # Truncate if sequence is longer than max_len
    return encoded[:max_len]

In [7]:
# Create word-to-index mapping from unique target words
word_to_idx = {}
words = train["target"].unique()
for i, word in enumerate(words):
    word_to_idx[word] = i

## 3. Data Encoding

Encode input letters and target words to numerical representations suitable for neural network training.

In [ ]:
# Encode training data
import ast

# Convert string representations of lists to actual lists and encode letters
train["encoded"] = train["input_letters"].apply(
    lambda x: encode_letters(ast.literal_eval(x))
)

# Map target words to their indices
train["label"] = train["target"].apply(
    lambda w: word_to_idx[w]
)

In [ ]:
# Import PyTorch and create custom Dataset class
import torch
from torch.utils.data import Dataset, DataLoader

class LetterWordDataset(Dataset):
    """Custom Dataset for letter-to-word prediction task"""
    def __init__(self, df):
        # X: encoded letter sequences as tensors
        self.X = torch.tensor(df["encoded"].tolist(), dtype=torch.long)
        # y: target word indices as tensors
        self.y = torch.tensor(df["label"].tolist(), dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
# Process validation and test sets
# Encode validation data
val["encoded"] = val["input_letters"].apply(
    lambda x: encode_letters(ast.literal_eval(x))
)
val["label"] = val["target"].apply(
    lambda w: word_to_idx[w]
)

# Encode test data
test["encoded"] = test["input_letters"].apply(
    lambda x: encode_letters(ast.literal_eval(x))
)
test["label"] = test["target"].apply(
    lambda w: word_to_idx[w]
)

# Create data loaders for batch processing
train_loader = DataLoader(LetterWordDataset(train), batch_size=64, shuffle=True)
val_loader = DataLoader(LetterWordDataset(val), batch_size=64)
test_loader = DataLoader(LetterWordDataset(test), batch_size=64)

## 4. Model Architecture

Define the LSTM neural network for sequence-to-word prediction.

In [ ]:
# Define LSTM model for letter-to-word prediction
import torch.nn as nn

class LetterToWordLSTM(nn.Module):
    """LSTM model: Embedding -> LSTM -> Fully Connected layer"""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_words):
        super().__init__()
        # Embedding layer: converts letter indices to dense vectors
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # LSTM layer: processes sequence of embeddings
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        # Fully connected layer: maps LSTM hidden state to word predictions
        self.fc = nn.Linear(hidden_dim, num_words)

    def forward(self, x):
        # Embed input letters
        x = self.embedding(x)
        # Pass through LSTM, get hidden state
        _, (h, _) = self.lstm(x)
        # Predict word class from final hidden state
        return self.fc(h[-1])

In [ ]:
# Initialize the model with hyperparameters
model = LetterToWordLSTM(
    vocab_size=len(letter_to_idx),  # 27 (26 letters + PAD)
    embed_dim=32,                     # Embedding dimension
    hidden_dim=128,                   # LSTM hidden dimension
    num_words=len(word_to_idx)        # Number of target words
)

## 5. Model Training

Train the LSTM model using cross-entropy loss and Adam optimizer.

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
for epoch in range(15):
    model.train()
    # Iterate through training batches
    for X, y in train_loader:
        # Zero the gradients from previous iteration
        optimizer.zero_grad()
        # Forward pass
        preds = model(X)
        # Compute loss
        loss = criterion(preds, y)
        # Backward pass
        loss.backward()
        # Update weights
        optimizer.step()

    print(f"Epoch {epoch+1} done")

Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done
Epoch 8 done
Epoch 9 done
Epoch 10 done
Epoch 11 done
Epoch 12 done
Epoch 13 done
Epoch 14 done
Epoch 15 done


## 6. Validation and Model Evaluation

Evaluate the trained model on the validation and test sets.

In [ ]:
# Evaluate model on validation set
model.eval()
correct = 0
with torch.no_grad():
    for X, y in val_loader:
        logits = model(X)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()

# Calculate validation accuracy
val_acc = correct / len(val)
print(f"Validation Accuracy: {val_acc:.4f}")

Validation Accuracy: 0.8116


In [ ]:
# Evaluate model on test set and compute confidence scores
model.eval()
correct = 0
with torch.no_grad():
    for X, y in test_loader:
        logits = model(X)
        # Get prediction confidence (softmax probabilities)
        probs = torch.softmax(logits, dim=1)
        confidence = torch.max(probs, dim=1).values
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()

# Calculate test accuracy
test_acc = correct / len(test)
print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 0.8089


In [ ]:
# Save model and vocab mappings
torch.save(model.state_dict(), "letter_to_word_model.pt")
torch.save(letter_to_idx, "letter_to_idx.pt")
torch.save(word_to_idx, "word_to_idx.pt")
print("Saved: letter_to_word_model.pt, letter_to_idx.pt, word_to_idx.pt")